# Ephemeral RayJob — distributed compute

Same ephemeral pattern as Topic 2: `RayJob` + `ManagedClusterConfig` via the CodeFlare SDK.

Runs `distributed_stats.py` with `@ray.remote` tasks across workers.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from workshop_bootstrap import setup_workshop_path

REPO_ROOT = setup_workshop_path()
print(f"Repo root: {REPO_ROOT}")

## Authenticate

Same credentials as Topic 2 — OpenShift Console → **Copy login command** → Display token.

In [ ]:
from codeflare_sdk import TokenAuthentication, list_local_queues

# Console → your username → Copy login command → Display token
OPENSHIFT_SERVER = "https://api.YOUR_CLUSTER:6443"
OPENSHIFT_TOKEN = "REPLACE_WITH_YOUR_TOKEN"

auth = TokenAuthentication(
    token=OPENSHIFT_TOKEN.strip(),
    server=OPENSHIFT_SERVER.strip(),
    skip_tls=True,
)
auth.login()

NAMESPACE = "ray-workshop"
LOCAL_QUEUE = "ray-workshop-queue"

list_local_queues(NAMESPACE)

In [ ]:
from codeflare_sdk import RayJob, ManagedClusterConfig

job = RayJob(
    job_name="ray-workshop-distributed-stats",
    entrypoint="python extras/scripts/distributed_stats.py",
    namespace=NAMESPACE,
    local_queue=LOCAL_QUEUE,
    cluster_config=ManagedClusterConfig(
        num_workers=2,
        head_cpu_requests=1,
        head_cpu_limits=2,
        head_memory_requests=2,
        head_memory_limits=4,
        worker_cpu_requests=1,
        worker_cpu_limits=2,
        worker_memory_requests=2,
        worker_memory_limits=4,
    ),
    runtime_env={
        "working_dir": str(REPO_ROOT),
        "env_vars": {
            "INPUT_PATH": "extras/data/iris.csv",
            "NUM_PARTITIONS": "4",
        },
    },
    ttl_seconds_after_finished=600,
)

job.submit()
print(f"Submitted: {job.job_name}")

In [ ]:
import time

terminal = {"SUCCEEDED", "FAILED", "STOPPED", "STOPPING"}
deadline = time.time() + 900

while time.time() < deadline:
    status = str(job.status())
    print(f"RayJob {job.job_name}: {status}")
    if status.upper() in terminal:
        break
    time.sleep(15)
else:
    raise TimeoutError(f"Timed out waiting for RayJob {job.job_name}")

print(job.logs())

## Check results

```sh
oc logs -n ray-workshop -l ray.io/node-type=head -c ray-head --tail=100 | tail -30
```

Expect JSON partition summaries and `Aggregated row count: 30`.